# WP-02 PoC — Instrument Extraction

**Goal:** Explore what instruments are present in each tango track.

**Two-tier design:**
- **Tier 1 — NMF decomposition** (always runs, zero new deps): sklearn NMF decomposes the spectrogram into N latent components, reconstructed as audio stems with soft Wiener masks.
- **Tier 2 — AudioSep zero-shot** (conditional, requires torch): text-guided separation using AudioSep. Cells skip gracefully if not installed — same pattern as the essentia cells in 02_audio_features.

**Tracks used:** `data/samples/` (3 tango samples, cortinas excluded)

---
## 0. Imports & Setup

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
import tempfile
from pathlib import Path
from sklearn.decomposition import NMF
from IPython.display import Audio, display

from atdj.config import SAMPLES_DIR, PROCESSED_DIR

STEMS_DIR = PROCESSED_DIR / "stems"
STEMS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Stems dir: {STEMS_DIR}")

# --- AudioSep setup ---
# AudioSep cannot be pip-installed. Clone it first:
#   git clone https://github.com/Audio-AGI/AudioSep.git vendor/AudioSep
# Then download checkpoints into vendor/AudioSep/checkpoint/:
#   music_speech_audioset_epoch_15_esc_89.98.pt  (CLAP encoder weights, ~2.2 GB)
#   audiosep_base_4M_steps.ckpt                  (main separation model, ~1.2 GB)
# Both at: https://huggingface.co/spaces/badayvedat/AudioSep/tree/main/checkpoint
#
_candidates = [
    Path("../vendor/AudioSep"),   # Jupyter launched from notebooks/
    Path("vendor/AudioSep"),      # Jupyter launched from project root
]
AUDIOSEP_REPO = next((p.resolve() for p in _candidates if p.exists()), None)
print(f"AudioSep repo: {AUDIOSEP_REPO}")

_orig_cwd = os.getcwd()
try:
    import torch
    import torchaudio
    if AUDIOSEP_REPO is None:
        raise ImportError("AudioSep repo not found — clone it to vendor/AudioSep (see Section 3)")

    # AudioSep loads CLAP checkpoint with a relative path at import time,
    # so cwd must be the repo root during import.
    os.chdir(AUDIOSEP_REPO)
    sys.path.insert(0, str(AUDIOSEP_REPO))
    from pipeline import build_audiosep, separate_audio as audiosep_separate
    AUDIOSEP_AVAILABLE = True
    print("AudioSep available — Tier 2 cells will run")
except Exception as e:
    AUDIOSEP_AVAILABLE = False
    print(f"AudioSep not available: {e}")
    print("See Section 3 for setup instructions.")
finally:
    os.chdir(_orig_cwd)

# --- Track list ---
# Tinta Roja is the preferred primary track for instrument extraction.
# Falls back to first available tango file if not present.
tango_files = sorted([f for f in SAMPLES_DIR.glob("*.mp3")
                      if "cortina" not in f.name.lower()])

_tinta = next((f for f in tango_files if "tinta roja" in f.name.lower()), None)
primary_track = _tinta if _tinta else tango_files[0]

print(f"\nTango sample files ({len(tango_files)}):")
for f in tango_files:
    marker = "  ◀ primary" if f == primary_track else ""
    print(f"  {f.name}{marker}")

---
## 1. Load Tracks + Quick Listen

Load all 3 tango samples. Play each one. Store `(y, sr)` pairs for later sections.

In [ ]:
tracks = {}  # dict: filename -> (y, sr)

print("=== Tango sample tracks ===\n")
for f in tango_files:
    y, sr = librosa.load(f, sr=None)
    tracks[f.name] = (y, sr)
    duration = len(y) / sr
    print(f"{f.name}  |  {sr} Hz  |  {duration:.1f}s")
    display(Audio(data=y, rate=sr))

---
## 2. NMF Source Decomposition (always runs)

Non-negative Matrix Factorization (NMF) decomposes the magnitude spectrogram `S ≈ W @ H` into N latent spectral components. Each component is reconstructed as audio using a soft Wiener mask (prevents hard clicks at component boundaries).

**Tango instrument stack** (what we are trying to find):
- **Contrabajo (double bass)** — low rhythmic pulse, fundamentals 60–250 Hz
- **Piano** — harmonic/rhythmic comping, body at 200–700 Hz but spans the full keyboard
- **Bandoneon** — the defining voice of tango, nasal wheeze concentrated at 300–1500 Hz
- **Violin section** — melody, often bright and present, 500–4000 Hz
- **Vocals** — when present, 200–3500 Hz (overlaps heavily with bandoneon and violin)

We use N=5 components — one per instrument. For vocal tracks a 6th component can help separate voice from violin; that's noted in Section 5.

**Caveat:** NMF has no notion of instrument names — it finds repeating spectral patterns. The labels below are frequency-based heuristics. On 1930s mono recordings these will blur: bandoneon and violin share a lot of frequency space, and piano spans almost everything.

### 2a. Decompose one track (El Retirado)

In [ ]:
y, sr = tracks[primary_track.name]

# STFT
D     = librosa.stft(y, n_fft=2048, hop_length=512)
S     = np.abs(D)
phase = np.angle(D)

# NMF decomposition
N_COMPONENTS = 5
nmf = NMF(n_components=N_COMPONENTS, max_iter=500, random_state=42)
W = nmf.fit_transform(S)   # shape: (freq_bins, N_COMPONENTS)
H = nmf.components_         # shape: (N_COMPONENTS, frames)

print(f"Track: {primary_track.name}")
print(f"STFT shape: {S.shape}  (freq_bins × frames)")
print(f"W shape: {W.shape}  (freq_bins × N_COMPONENTS)")
print(f"H shape: {H.shape}  (N_COMPONENTS × frames)")

In [ ]:
# Reconstruct each component using soft Wiener mask
S_approx = W @ H  # full NMF reconstruction
freqs = librosa.fft_frequencies(sr=sr, n_fft=2048)

components = []
centroids  = []

for i in range(N_COMPONENTS):
    S_comp = np.outer(W[:, i], H[i])
    mask   = S_comp / (S_approx + 1e-8)   # soft mask, values 0–1
    D_comp = S * mask * np.exp(1j * phase)
    y_comp = librosa.istft(D_comp, hop_length=512)
    components.append(y_comp)

    # Spectral centroid of this component's basis spectrum W[:,i]
    cent = float(np.sum(freqs * W[:, i]) / (np.sum(W[:, i]) + 1e-8))
    centroids.append(cent)

    # Plot spectrogram
    fig, ax = plt.subplots(figsize=(10, 2.5))
    S_db = librosa.amplitude_to_db(S_comp, ref=np.max)
    img = librosa.display.specshow(S_db, sr=sr, hop_length=512,
                                   x_axis='time', y_axis='log', ax=ax)
    ax.set_title(f"Component {i+1}  |  centroid = {cent:.0f} Hz")
    plt.colorbar(img, ax=ax, format='%+2.0f dB')
    plt.tight_layout()
    plt.show()

    print(f"  Component {i+1}: centroid = {cent:.0f} Hz")
    display(Audio(y_comp, rate=sr))
    print()

### 2b. Heuristic frequency labeling

Sort components by spectral centroid to assign a tango instrument guess:

| Centroid range | Tango instrument | Why |
|---|---|---|
| < 300 Hz | contrabajo (double bass) | Lowest fundamentals, rhythmic pulse |
| 300–700 Hz | piano / bandoneon body | Piano left-hand comping; bandoneon's chest register |
| 700–1500 Hz | bandoneon upper / violin lower | Bandoneon melody range; violin's lower strings |
| 1500–3000 Hz | violin melody / vocals | Violin in upper register; voice formants |
| > 3000 Hz | violin harmonics / treble | Upper partials, bow noise, high overtones |

**Note:** these are heuristics — NMF finds repeating spectral patterns, not instrument names. Listen to the audio players in 2a to judge whether the label matches what you actually hear.

In [ ]:
def centroid_to_label(hz: float) -> str:
    """Map spectral centroid to a tango instrument guess based on typical frequency ranges."""
    if hz < 300:
        return "contrabajo (double bass)"
    elif hz < 700:
        return "piano / bandoneon body"
    elif hz < 1500:
        return "bandoneon upper / violin lower"
    elif hz < 3000:
        return "violin melody / vocals"
    else:
        return "violin harmonics / treble"

# Sort by centroid and print labeled table
order = np.argsort(centroids)
print(f"NMF component labels — {tango_files[0].name}\n")
print(f"{'Rank':<5} {'Component':<12} {'Centroid (Hz)':<16} {'Tango instrument guess'}")
print("-" * 65)
for rank, idx in enumerate(order):
    print(f"{rank+1:<5} {idx+1:<12} {centroids[idx]:<16.0f} {centroid_to_label(centroids[idx])}")

print()
print("Listen to the audio players above to check whether each label matches what you hear.")

### 2c. NMF across all 3 tango samples

Output: track × component energy DataFrame, sorted by centroid. One bar chart per track.

In [ ]:
all_nmf_results = []

for f in tango_files:
    y_t, sr_t = tracks[f.name]
    D_t     = librosa.stft(y_t, n_fft=2048, hop_length=512)
    S_t     = np.abs(D_t)
    freqs_t = librosa.fft_frequencies(sr=sr_t, n_fft=2048)

    nmf_t = NMF(n_components=N_COMPONENTS, max_iter=500, random_state=42)
    W_t   = nmf_t.fit_transform(S_t)
    H_t   = nmf_t.components_
    S_approx_t = W_t @ H_t

    row = {"track": f.name}
    for i in range(N_COMPONENTS):
        S_comp_i = np.outer(W_t[:, i], H_t[i])
        energy_i = float(np.mean(S_comp_i))
        cent_i   = float(np.sum(freqs_t * W_t[:, i]) / (np.sum(W_t[:, i]) + 1e-8))
        label_i  = centroid_to_label(cent_i)
        row[f"comp{i+1}_centroid_hz"] = round(cent_i, 0)
        row[f"comp{i+1}_energy"]      = round(energy_i, 4)
        row[f"comp{i+1}_label"]       = label_i
    all_nmf_results.append(row)

    # Bar chart: component energy profile for this track
    energies  = [row[f"comp{i+1}_energy"]      for i in range(N_COMPONENTS)]
    cents     = [row[f"comp{i+1}_centroid_hz"]  for i in range(N_COMPONENTS)]
    labels    = [row[f"comp{i+1}_label"]        for i in range(N_COMPONENTS)]
    bar_labels = [f"C{i+1}\n{cents[i]:.0f}Hz" for i in range(N_COMPONENTS)]

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.bar(bar_labels, energies, color='steelblue')
    ax.set_title(f"NMF Component Energy — {f.name}")
    ax.set_ylabel("Mean energy")
    ax.set_xlabel("Component (centroid Hz)")
    plt.tight_layout()
    plt.show()

df_nmf = pd.DataFrame(all_nmf_results)
print("\nNMF summary (centroids):")
centroid_cols = ["track"] + [f"comp{i+1}_centroid_hz" for i in range(N_COMPONENTS)]
display(df_nmf[centroid_cols])

---
## 3. AudioSep Zero-Shot Separation (conditional)

> **Mac/Linux / GPU users:** run these cells after setting up AudioSep (instructions below).
> Windows CPU: works but slow (~10 min/track). GPU strongly recommended.
> If not set up, cells skip cleanly with a printed message.

### Setting up AudioSep

AudioSep has no `setup.py` so `pip install git+...` fails. You need to clone it manually.

**Step 1 — install PyTorch:**

CPU-only (Windows, ~600 MB):
```
pip install torch torchaudio --index-url https://download.pytorch.org/whl/cpu
```

GPU (if CUDA available):
```
pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121
```

**Step 2 — clone the repo and install its dependencies:**
```
git clone https://github.com/Audio-AGI/AudioSep.git vendor/AudioSep
pip install -r vendor/AudioSep/requirements.txt
```

**Step 3 — set the path in the cell below** (`AUDIOSEP_REPO` variable) to wherever you cloned it, then restart the kernel and re-run from Section 0.

Note: Python 3.13 requires PyTorch ≥ 2.5. Model weights (~1 GB) download on first use.

In [ ]:
# AudioSep dependency installs — run these once, then restart the kernel.
#
# Step 1: PyTorch CPU (~600 MB). Skip if already installed.
# !pip install torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cpu
#
# Step 2: Clone the AudioSep repo (no setup.py, cannot pip install directly)
# !git clone https://github.com/Audio-AGI/AudioSep.git ../vendor/AudioSep
#
# Step 3: All missing packages discovered by tracing AudioSep imports:
#   - lightning      : pl.LightningModule base class (audiosep.py)
#   - transformers   : RobertaTokenizer (clap_encoder.py)
#   - torchlibrosa   : STFT/ISTFT ops (resunet.py)
#   - huggingface_hub: PyTorchModelHubMixin (audiosep.py)
#   - torchvision    : FrozenBatchNorm2d (CLAP/open_clip/utils.py)
#   - h5py           : audio feature storage (CLAP/open_clip/utils.py)
#   - ftfy           : text normalisation (CLAP/open_clip/tokenizer.py)
#   - braceexpand    : dataset path expansion (CLAP/training/data.py)
#   - webdataset     : streaming dataset loader (CLAP/training/data.py)
#   - timm           : vision transformer backbone (CLAP/open_clip/timm_model.py)
#   - pyloudnorm     : loudness normalisation
#   - wget           : checkpoint download utility
# !pip install lightning transformers torchlibrosa huggingface_hub torchvision h5py ftfy braceexpand webdataset timm pyloudnorm wget
#
# Step 4: Download checkpoints into vendor/AudioSep/checkpoint/
#   (both ~500 MB each, from https://huggingface.co/spaces/badayvedat/AudioSep/tree/main/checkpoint)
# !wget "https://huggingface.co/spaces/badayvedat/AudioSep/resolve/main/checkpoint/music_speech_audioset_epoch_15_esc_89.98.pt" -O ../vendor/AudioSep/checkpoint/music_speech_audioset_epoch_15_esc_89.98.pt
# !wget "https://huggingface.co/spaces/badayvedat/AudioSep/resolve/main/checkpoint/audiosep_base_4M_steps.ckpt" -O ../vendor/AudioSep/checkpoint/audiosep_base_4M_steps.ckpt
print("Run the commented steps above if AudioSep is not available.")

### 3a. Load AudioSep model

In [ ]:
if AUDIOSEP_AVAILABLE:
    import torch
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    CKPT_PATH = AUDIOSEP_REPO / "checkpoint" / "audiosep_base_4M_steps.ckpt"
    CONFIG_PATH = AUDIOSEP_REPO / "config" / "audiosep_base.yaml"

    if not CKPT_PATH.exists():
        print(f"ERROR: checkpoint not found at {CKPT_PATH}")
        print("Download from: https://huggingface.co/spaces/badayvedat/AudioSep/tree/main/checkpoint")
        AUDIOSEP_AVAILABLE = False
    else:
        _orig_cwd = os.getcwd()
        os.chdir(AUDIOSEP_REPO)
        model = build_audiosep(
            config_yaml=str(CONFIG_PATH),
            checkpoint_path=str(CKPT_PATH),
            device=device,
        )
        os.chdir(_orig_cwd)
        print("AudioSep model loaded")
else:
    print("Skipping — AudioSep not available")

### 3b. Separate one track with 5 text queries

In [ ]:
QUERIES = {
    "bandoneon": "bandoneon accordion",
    "violin":    "violin strings",
    "piano":     "piano melody",
    "bass":      "double bass",
    "vocals":    "singing voice",
}

if AUDIOSEP_AVAILABLE:
    track = primary_track
    y_as, sr_as = librosa.load(track, sr=32000, mono=True)
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        sf.write(f.name, y_as, sr_as)
        wav_in = f.name
    print(f"Track: {track.name}\n")

    for name, query in QUERIES.items():
        wav_out = str(STEMS_DIR / f"{track.stem}_{name}.wav")
        audiosep_separate(model, wav_in, query, wav_out, device)
        y_sep, _ = librosa.load(wav_out, sr=sr_as)
        energy = float(np.mean(librosa.feature.rms(y=y_sep)))
        print(f"  [{name}]  query='{query}'  energy={energy:.4f}")
        display(Audio(y_sep, rate=sr_as))
        print()
else:
    print("Skipping — AudioSep not available")

### 3c. Presence matrix across all 3 tracks

For each track × query: record RMS energy of separated stem. `PRESENCE_THRESHOLD = 0.01` — above this, instrument is considered "present".

In [ ]:
PRESENCE_THRESHOLD = 0.01

if AUDIOSEP_AVAILABLE:
    presence_rows = []
    for f in tango_files:
        y_as, sr_as = librosa.load(f, sr=32000, mono=True)
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
            sf.write(tmp.name, y_as, sr_as)
            wav_in = tmp.name

        row = {"track": f.name}
        for name, query in QUERIES.items():
            wav_out = str(STEMS_DIR / f"{f.stem}_{name}.wav")
            audiosep_separate(model, wav_in, query, wav_out, device)
            y_sep, _ = librosa.load(wav_out, sr=sr_as)
            row[name] = round(float(np.mean(librosa.feature.rms(y=y_sep))), 4)
        presence_rows.append(row)

    df_presence = pd.DataFrame(presence_rows).set_index("track")
    print("RMS energy per instrument per track:")
    display(df_presence)

    fig, ax = plt.subplots(figsize=(8, 3))
    im = ax.imshow(df_presence.values, aspect='auto', cmap='YlOrRd')
    ax.set_xticks(range(len(QUERIES)))
    ax.set_xticklabels(list(QUERIES.keys()))
    ax.set_yticks(range(len(tango_files)))
    ax.set_yticklabels([f.name[:25] for f in tango_files], fontsize=8)
    plt.colorbar(im, ax=ax, label='RMS energy')
    ax.set_title('Instrument presence heatmap (AudioSep)')
    plt.tight_layout()
    plt.show()
else:
    print("Skipping — AudioSep not available")

---
## 4. Presence Detection Summary

In [ ]:
if AUDIOSEP_AVAILABLE:
    # Binary presence table from AudioSep energies
    binary = df_presence.applymap(lambda x: '✓' if x >= PRESENCE_THRESHOLD else '—')
    print(f"Instrument presence (threshold = {PRESENCE_THRESHOLD})")
    print()
    display(binary)

    print()
    print("Cross-check note: if 'vocals' shows ✓ but the ID3 tag says 'Instrumental',")
    print("the tag may be incorrect. Compare with version_vocal field from notebook 02_audio_features.")
else:
    # Fallback: show NMF-derived component count per track
    print("AudioSep not available — showing NMF component summary instead")
    print()
    nmf_summary_rows = []
    for f in tango_files:
        y_t, sr_t = tracks[f.name]
        D_t   = librosa.stft(y_t, n_fft=2048, hop_length=512)
        S_t   = np.abs(D_t)
        freqs_t = librosa.fft_frequencies(sr=sr_t, n_fft=2048)
        nmf_t = NMF(n_components=N_COMPONENTS, max_iter=500, random_state=42)
        W_t   = nmf_t.fit_transform(S_t)
        H_t   = nmf_t.components_

        label_counts = {}
        for i in range(N_COMPONENTS):
            cent_i = float(np.sum(freqs_t * W_t[:, i]) / (np.sum(W_t[:, i]) + 1e-8))
            lbl    = centroid_to_label(cent_i)
            label_counts[lbl] = label_counts.get(lbl, 0) + 1

        row = {"track": f.name}
        row.update(label_counts)
        nmf_summary_rows.append(row)

    df_nmf_summary = pd.DataFrame(nmf_summary_rows).fillna(0).set_index("track")
    print(f"NMF components per frequency band (N={N_COMPONENTS} total):")
    display(df_nmf_summary)